# App review sentiment classification: review notebook

New portfolio walkthrough based on the ProjectPro learning objective, with assistant-supported implementation. This is classical NLP, not an LLM. See `docs/SOURCE_AND_CHANGES.md` for source credit and changes.

Set the environment variable `DATASET_PATH` to your authorized local `Canva_reviews.xlsx` before starting the notebook. Do not upload the workbook or personal reviewer metadata. No API keys or downloaded model binaries are required.


In [1]:
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "Engine.py").is_file(), "Start Jupyter in the repository root or notebooks folder."
sys.path.insert(0, str(root))
import pandas as pd
from Source.model import prepare_data, split_reviews, build_pipeline
from Engine import evaluate
from predict import predict
raw_path = os.environ.get("DATASET_PATH")
if not raw_path:
    raise ValueError("Set DATASET_PATH to your authorized local dataset.")
path = Path(raw_path).expanduser().resolve()
frame = pd.read_excel(path, usecols=["review", "Sentiment"])
print({"rows": len(frame), "labels": frame.Sentiment.value_counts().to_dict()})


{'rows': 1500, 'labels': {'Positive': 1032, 'Negative': 468}}


## 1. Validate and split before learning text features

Only the two modeling columns are loaded. Deterministic blank/duplicate checks are applied before a stratified 80/20 split. Exact normalized duplicates cannot appear on both sides. This does not rule out near duplicates or reviewer/time correlation.


In [2]:
data, audit = prepare_data(frame)
train_set, test_set = split_reviews(data, test_size=0.2, seed=42)
print(audit)
print({"train_rows": len(train_set), "test_rows": len(test_set), "text_overlap": len(set(train_set.normalized) & set(test_set.normalized))})
print({"training_review_length_median": float(train_set.review.str.len().median())})


{'input_rows': 1500, 'blank_normalized_rows_removed': 0, 'conflicting_text_rows_removed': 0, 'duplicate_text_rows_removed': 0, 'usable_rows': 1500, 'label_counts': {'Positive': 1032, 'Negative': 468}}
{'train_rows': 1200, 'test_rows': 300, 'text_overlap': 0}
{'training_review_length_median': 76.0}


## 2. Train four fixed representations

The pipeline fits vocabulary and IDF on training text only. Stopwords and negation are retained; no stemming is used. This differs from the supplied tutorial and is not a controlled before/after leakage experiment. TF-IDF is fixed as the default before viewing scores. The comparison below is exploratory; do not tune on this holdout and then claim unbiased final performance.


In [3]:
models = {}
results = []
for mode in ("bow", "bowb", "ng", "tf"):
    pipeline = build_pipeline(mode, min_df=5, seed=42)
    pipeline.fit(train_set.review, train_set.Sentiment)
    scores = evaluate(test_set.Sentiment, pipeline.predict(test_set.review))
    models[mode] = pipeline
    results.append({"mode": mode, "accuracy": scores["accuracy"], "balanced_accuracy": scores["balanced_accuracy"], "macro_f1": scores["macro_f1"]})
print(pd.DataFrame(results).to_string(index=False))


mode  accuracy  balanced_accuracy  macro_f1
 bow  0.916667           0.910401  0.904521
bowb  0.913333           0.905082  0.900429
  ng  0.916667           0.913293  0.905033
  tf  0.940000           0.924499  0.929452


## 3. Compare the majority baseline and inspect aggregate errors

Accuracy alone hides class imbalance. Macro F1 and per-class precision/recall are useful alongside a majority baseline. No individual reviews or identities are displayed.


In [4]:
from sklearn.dummy import DummyClassifier
import numpy as np
baseline = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(train_set), 1)), train_set.Sentiment)
print(evaluate(test_set.Sentiment, baseline.predict(np.zeros((len(test_set), 1)))))
print(evaluate(test_set.Sentiment, models["tf"].predict(test_set.review)))


{'accuracy': 0.6866666666666666, 'balanced_accuracy': 0.5, 'macro_f1': 0.40711462450592883, 'confusion_matrix_label_order': ['Negative', 'Positive'], 'confusion_matrix': [[0, 94], [0, 206]], 'classification_report': {'Negative': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 94.0}, 'Positive': {'precision': 0.6866666666666666, 'recall': 1.0, 'f1-score': 0.8142292490118577, 'support': 206.0}, 'accuracy': 0.6866666666666666, 'macro avg': {'precision': 0.3433333333333333, 'recall': 0.5, 'f1-score': 0.40711462450592883, 'support': 300.0}, 'weighted avg': {'precision': 0.47151111111111105, 'recall': 0.6866666666666666, 'f1-score': 0.5591040843214756, 'support': 300.0}}}
{'accuracy': 0.94, 'balanced_accuracy': 0.9244990704399918, 'macro_f1': 0.9294523411371238, 'confusion_matrix_label_order': ['Negative', 'Positive'], 'confusion_matrix': [[83, 11], [7, 199]], 'classification_report': {'Negative': {'precision': 0.9222222222222223, 'recall': 0.8829787234042553, 'f1-score': 0.902

## 4. Predict on authored example text

The same pipeline handles preprocessing and features. The probabilities are uncalibrated model outputs, not certainty. An unknown-vocabulary input gets an explicit warning. These examples are authored smoke tests, not copied dataset rows.


In [5]:
for text in ["The design editor is useful and easy to use", "The app crashes and does not save my work", "zzzzzzz"]:
    print(predict(models["tf"], text))


{'label': 'Positive', 'class_probabilities': {'Negative': 0.0741131971690333, 'Positive': 0.9258868028309667}, 'known_feature_count': 9, 'warning': None, 'note': 'Model probabilities are not calibrated confidence guarantees.'}
{'label': 'Negative', 'class_probabilities': {'Negative': 0.8718334143974172, 'Positive': 0.1281665856025828}, 'known_feature_count': 8, 'warning': None, 'note': 'Model probabilities are not calibrated confidence guarantees.'}
{'label': 'Positive', 'class_probabilities': {'Negative': 0.4022651095564801, 'Positive': 0.5977348904435199}, 'known_feature_count': 0, 'warning': 'No known features; prediction is driven by the intercept.', 'note': 'Model probabilities are not calibrated confidence guarantees.'}


## 5. Explain this project in a review

- Why must the split happen before fitting the vectorizer?
- Why can retaining “not” matter for sentiment?
- What does the majority baseline tell us about accuracy?
- What remains untested: near-duplicate leakage, calibration, temporal drift, sarcasm, multilingual reviews and production behavior?
- Which changes did you personally review and understand?

Original teaching files remain separate; this notebook does not claim independent authorship of the underlying course project. Do not assert deployment or business impact that did not occur.
